# Implementasi WAW Saja pada Simulator ILP
Notebook ini mempertahankan struktur simulator dosen, tetapi **hanya mendeteksi Write After Write (WAW)**. Instruksi contoh dipilih agar tidak muncul RAW atau WAR. Semua instruksi diasumsikan satu siklus, dan penjadwalan ini adalah demonstrasi dependensi WAW, bukan simulator hazard lengkap.


In [ ]:
# SIMULATOR ILP - KHUSUS DEPENDENSI WAW (Write After Write)
from dataclasses import dataclass
import matplotlib.pyplot as plt
import networkx as nx

@dataclass
class Instruction:
    name: str
    destination: str
    sources: list

# Hanya terdapat dependensi WAW: I1 -> I2 dan I2 -> I4.
# Register sumber tidak ditulis oleh instruksi mana pun dalam contoh ini.
instructions = [
    Instruction("I1", "R1", ["R2", "R3"]),
    Instruction("I2", "R1", ["R4", "R5"]),
    Instruction("I3", "R6", ["R7", "R8"]),
    Instruction("I4", "R1", ["R9", "R10"]),
    Instruction("I5", "R11", ["R12", "R13"]),
]

def collect_waw_dependencies(instructions):
    last_writer = {}
    edges = []
    for inst in instructions:
        if inst.destination in last_writer:
            edges.append((last_writer[inst.destination].name, inst.name, inst.destination))
        last_writer[inst.destination] = inst
    return edges

def show_dependencies(instructions):
    print("\nWAW DEPENDENCY ANALYSIS")
    edges = collect_waw_dependencies(instructions)
    if not edges:
        print("Tidak ada dependensi WAW.")
    for before, after, register in edges:
        print(f"{before} -> {after}: WAW (keduanya menulis {register})")
    return edges

def draw_dependency_graph(instructions):
    edges = collect_waw_dependencies(instructions)
    graph = nx.DiGraph()
    graph.add_nodes_from(inst.name for inst in instructions)
    graph.add_edges_from((a, b) for a, b, _ in edges)
    pos = {inst.name: (i % 3, -(i // 3)) for i, inst in enumerate(instructions)}
    labels = {inst.name: f"{inst.name}\n{inst.destination} = {' + '.join(inst.sources)}" for inst in instructions}
    plt.figure(figsize=(10, 5))
    nx.draw_networkx_nodes(graph, pos, node_size=2900, node_color="#dbeafe")
    nx.draw_networkx_labels(graph, pos, labels=labels, font_size=8)
    nx.draw_networkx_edges(graph, pos, arrows=True, arrowsize=20, node_size=2900, edge_color="#b91c1c", width=2)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels={(a,b):f"WAW ({r})" for a,b,r in edges}, font_size=9)
    plt.title("Dependency Graph - WAW Only")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def sequential_execution(instructions):
    print("\nSEQUENTIAL EXECUTION")
    for cycle, inst in enumerate(instructions, 1):
        print(f"Cycle {cycle}: {inst.name} -> {inst.destination}")
    return len(instructions)

def parallel_execution(instructions):
    edges = collect_waw_dependencies(instructions)
    predecessors = {inst.name: set() for inst in instructions}
    for before, after, _ in edges:
        predecessors[after].add(before)
    remaining = instructions.copy()
    completed = set()
    cycle = 0
    print("\nILP PARALLEL EXECUTION (WAW-ONLY MODEL)")
    while remaining:
        ready = [inst for inst in remaining if predecessors[inst.name] <= completed]
        if not ready:
            raise RuntimeError("Tidak ada instruksi siap dijalankan.")
        cycle += 1
        print(f"Cycle {cycle}: " + ", ".join(inst.name for inst in ready))
        completed.update(inst.name for inst in ready)
        remaining = [inst for inst in remaining if inst.name not in completed]
    return cycle

def main():
    print("SIMULATOR ILP - IMPLEMENTASI WAW SAJA")
    print("\nInstruction Set:")
    for inst in instructions:
        print(f"{inst.name}: {inst.destination} = {' + '.join(inst.sources)}")
    show_dependencies(instructions)
    draw_dependency_graph(instructions)
    seq = sequential_execution(instructions)
    par = parallel_execution(instructions)
    print("\nPERFORMANCE COMPARISON")
    print(f"Sequential Cycles: {seq}")
    print(f"ILP Cycles: {par}")
    print(f"Speedup: {seq/par:.2f}x")
    print("Catatan: model ini sengaja hanya memeriksa WAW; bukan simulator hazard lengkap.")

main()
